In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # put this at the very top of your script

from datasets import load_dataset, Dataset
from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from tools_llada import TopKSorter, TruthCollector, MaxCollector
from plugins_llada import \
    SaveKVPreviousPlugin_Disabled, SaveKVPreviousPlugin_Enabled,\
    CachePastKVPlugin_Disabled, CachePastKVPlugin_Enabled,\
    CacheAttnPlugin_Disabled, CacheAttnPlugin_Enabled,\
    CacheVOPlugin_Disabled, CacheVOPlugin_Enabled

from configs_llada import DiffusionConfig

from components_llada import SimpleLogitsSnapshot
from tools_llada import ConfKSorter, ConfCollectorInterface, BlockDiffusionQuotaHelper
from plugins_llada import InspectorPlugin

from dataset_llada import LIST_DATASET
from datapreprocess_llada import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI
from dataprocess_llada import Tokenizer_wiki_simple, Collater_wiki_simple

from modeling_llada_yukai_v4 import LLaDAModelLM

from tools_debug import jprint


config = DiffusionConfig(
    id_model='GSAI-ML/LLaDA-8B-Base',
    len_prompt=128,
    len_target=256,
    num_blocks=1,
    num_unmask_per_step=1,
    id_mask=126336,
    size_batch=1,
    device='cuda:0',
    klass_sorter=TopKSorter,
    klass_collector=TruthCollector,
    klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
    klass_cache_past_kv=CachePastKVPlugin_Enabled,
    klass_cache_attn=CacheAttnPlugin_Disabled,
    klass_cache_vo=CacheVOPlugin_Disabled
)

/home/exx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

'''load dataset first'''
name_dataset = LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:100])


'''initialize constant hyper-parameters'''

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    config.id_model,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336


'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    config.id_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(config.device)


'''start to handle dataset based on hyper-parameter'''
len_max = config.len_prompt + config.len_target
ds = ds_origin\
    .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
    .map(Tokenizer_wiki_simple(tokenizer, len_max), remove_columns=ds_origin.column_names)\
    .filter(lambda x: x["length"] >= len_max)\
    .sort("length")
# end

'''prepare dataloader'''
loader = DataLoader(
    ds,
    batch_size=config.size_batch,
    shuffle=False,                 # keep sorted order
    collate_fn=Collater_wiki_simple(len_max, config.len_prompt, config.len_target, config.id_mask),
    drop_last=False
)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Filter: 100%|██████████| 100/100 [00:00<00:00, 16083.07 examples/s]


In [3]:
@ torch.no_grad()
def run_model_semi_cached_refresh(model, x, y, config_diffusion, *args, **kwargs):

    '''declare required variables'''
    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.klass_sorter()
    collector = config_diffusion.klass_collector()
    refresher = kwargs['refresher']

    p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
    idx_denoising = torch.arange(0, len_prompt, dtype=torch.long).to(x.device)
    model(x[:, idx_denoising], idx_current=idx_denoising)   # save prompt for once

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,position_start:position_end] == id_mask
        idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)
        idx_refresh = refresher.get_refresh_idx(x, 0, id_block, return_sorted=True) # 4.87 if idx_refresh get not block, but first step

        for step in range(step_per_block):

            # idx_refresh = refresher.get_refresh_idx(x, step, id_block, return_sorted=True)
            idx_current = torch.cat([idx_refresh, idx_denoising])
            shape_target = (x.shape[0], position_end, -1)
            x_current, x_denoising,  y_denoising= x[:, idx_current], x[:, idx_denoising], y[:, idx_denoising]

            logits_current = model(x_current, idx_current=idx_current, shape_target=shape_target).logits
            logits_denoising = logits_current[:, -idx_denoising.shape[-1]:]
            snapshot = SimpleLogitsSnapshot(logits_denoising, x_denoising, y_denoising, id_mask)
            
            conf_snapshot = snapshot.transform_logits(collector)
            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)
            idx_transform = idx_sorted_by_conf[:, :num_unmask]

            snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
            snapshot.update_this(1, idx_src=idx_transform, idx_tgt=idx_denoising, y=x).update_this(1, idx_src=idx_transform, idx_tgt=idx_denoising, p_finalized=p_finalized)
        # end for step
    # end for block

    return p_finalized[:, len_prompt:]
# end

In [4]:
import json
from tqdm import tqdm
from tools_llada import PPLCalculator, RefreshIdxHelper

filename = 'all_in_one_diff_128_256_4_abs_per_block_p_0326.json'
with open(filename, 'r') as file:
    data_refresh = json.load(file)
# end

refresher = RefreshIdxHelper(
    data_refresh,
    'v',
    config.size_block,
    randomed=False
)

calculator_ppl = PPLCalculator()
model\
    .fill_plugin(config.klass_cache_past_kv)\
    .fill_plugin(config.klass_save_kv_previous)\
    .fill_plugin(config.klass_cache_attn)\
    .fill_plugin(config.klass_cache_vo)

plugin_cache_past_kv = config.klass_cache_past_kv()

for budget in (1,48):

    '''start the evaluation process'''
    for id_batch, batch in enumerate(tqdm(loader)):
        plugin_cache_past_kv.clear(model)
        refresher.set_sample_id(id_batch).set_budget(budget)

        conf = run_model_semi_cached_refresh(
            model,
            batch['ids_prompt_masked_full'].to(config.device),
            batch['ids_target_masked_full'].to(config.device),
            config,
            refresher=refresher
        )

        print(calculator_ppl.cal(conf))
        break
    # end for
# end

  0%|          | 0/92 [00:14<?, ?it/s]


(8.056853550258927, 0.37921547706740727)


  0%|          | 0/92 [00:13<?, ?it/s]

(7.718535846276374, 0.38357841615342925)
